# Aprendizado de Máquina — Lista prática 01

## Introdução ao Aprendizado de Máquina

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Na aula prática vimos que o erro de treino desce sempre e que o risco tem forma
de U. Aqui você vai refazer aquilo por conta própria, numa amostra nova, e a
lista termina numa pergunta que a Aula 01 não resolve:

> **se eu escolher o grau olhando o erro de teste de uma única amostra, quão
> confiável é essa escolha?**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula. As lacunas estão numeradas nos comentários, `# (a)`, `# (b)`, e
assim por diante.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
from matplotlib.pyplot import subplots

import sklearn.linear_model as skl
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

---
## Exercício 1 — a população

A população é a mesma da Aula 01:

$$X \sim \mathrm{Unif}[-3,3], \qquad
  Y = \underbrace{\sin(1{,}5X) + 0{,}3X}_{r(X)} + \varepsilon, \qquad
  \varepsilon \sim N(0;\,0{,}7^2).$$

Complete a função de regressão e o sorteio da amostra. Note que $r$ recebe um
vetor e devolve um vetor — nada de laço.

In [ ]:
A, B = -3.0, 3.0
SIGMA = 0.7


def r(x):
    """Função de regressão verdadeira, r(x) = E[Y | X = x]."""
    return np.sin(1.5 * x) + 0.3 * x      # (a)


def amostra(n, rng):
    """Sorteia n pares (x, y) da população."""
    x = rng.uniform(A, B, size=n)                     # (b)
    y = r(x) + rng.normal(0, SIGMA, size=n)           # (c)
    return x, y

Confira o sorteio antes de seguir. O desvio-padrão de $Y$ tem de ficar bem acima
de $\sigma = 0{,}7$: além do ruído, $Y$ varia porque $r(X)$ varia.

In [ ]:
rng = np.random.default_rng(2026)
x_tr, y_tr = amostra(50, rng)
x_te, y_te = amostra(2000, rng)

print(f"treino: n = {len(x_tr)}, dp de y = {y_tr.std(ddof=1):.4f}")
print(f"teste:  n = {len(x_te)}")

Deve imprimir `treino: n = 50, dp de y = 1.1218` e `teste: n = 2000`.

O desvio-padrão $1{,}12$ contra $\sigma = 0{,}7$ diz que a maior parte da
variação de $Y$ vem de $r(X)$, não do ruído — é por isso que vale a pena tentar
estimar $r$.

> **Sua vez.** Desenhe a nuvem de treino e, por cima, a curva $r$ verdadeira numa
> grade fina de $x$. É a única vez no curso em que você pode fazer isso: numa
> população real, $r$ é justamente o que não se conhece.

---
## Exercício 2 — três graus, dois erros

Ajuste polinômios de grau 1, 5 e 15 à **mesma** amostra de treino e meça o erro
quadrático médio nos dois conjuntos.

O `Pipeline` abaixo já está montado: ele cria as potências de $x$, padroniza (para
o ajuste não sofrer com $x^{15}$ ser enorme) e resolve por mínimos quadrados.

In [ ]:
def modelo_poly(grau):
    return Pipeline([
        ("poly", PolynomialFeatures(degree=grau, include_bias=False)),
        ("escala", StandardScaler()),
        ("mqo", skl.LinearRegression()),
    ])


X_tr = x_tr.reshape(-1, 1)
X_te = x_te.reshape(-1, 1)

for grau in [1, 5, 15]:                                     # (a)
    modelo = modelo_poly(grau).fit(X_tr, y_tr)            # (b) ajuste no treino
    erro_tr = np.mean((y_tr - modelo.predict(X_tr)) ** 2)
    erro_te = np.mean((y_te - modelo.predict(X_te)) ** 2)   # (c) o mesmo, no teste
    print(f"grau {grau:2d}:  treino {erro_tr:.4f}   teste {erro_te:.4f}")

Deve imprimir:

```
grau  1:  treino 0.9271   teste 1.0338
grau  5:  treino 0.5554   teste 0.6894
grau 15:  treino 0.4239   teste 67511.6003
```

Os três casos da aula, em três linhas: o grau 1 erra parecido nos dois conjuntos
(subajuste — o modelo é rígido demais para se aproveitar do treino), o grau 5
tem os dois erros baixos, e o grau 15 tem o **menor** erro de treino de todos e um
erro de teste cinco ordens de grandeza maior.

**Responda:** o grau 15 é o que menos erra no treino. Por que isso não é motivo
nenhum para escolhê-lo? Escreva a resposta na célula abaixo, como comentário.

Porque o erro de treino não é o risco. O polinômio de grau 15 tem 15 coeficientes
para ajustar 50 pontos: ele consegue passar perto de cada ponto observado,
inclusive acompanhando o **ruído** de cada um. Isso derruba o erro de treino e
destrói o desempenho fora da amostra, porque o ruído da próxima observação será
outro. O erro de treino é otimista por construção — o modelo foi escolhido para
ir bem justamente naqueles pontos.

---
## Exercício 3 — a curva em U

Agora varra os graus de 1 a 10, guarde os dois erros e desenhe as duas curvas.
Use escala logarítmica no eixo vertical: sem ela, o grau 10 achata todo o resto.

In [ ]:
graus = np.arange(1, 11)                                    # (a)
erros_tr, erros_te = [], []

for grau in graus:
    modelo = modelo_poly(grau).fit(X_tr, y_tr)
    erros_tr.append(np.mean((y_tr - modelo.predict(X_tr)) ** 2))
    erros_te.append(np.mean((y_te - modelo.predict(X_te)) ** 2))

erros_tr, erros_te = np.array(erros_tr), np.array(erros_te)
melhor = graus[int(np.argmin(erros_te))]                    # (b) o grau de menor erro de teste
print(f"menor erro de teste: grau {melhor}  ({erros_te.min():.4f})")

In [ ]:
fig, ax = subplots(figsize=(5, 3.2))
ax.plot(graus, erros_tr, "s-", ms=4, label="erro de treino")
ax.plot(graus, erros_te, "o-", ms=4, label="erro de teste")
ax.axhline(SIGMA ** 2, ls="--", lw=1.2, color="gray",       # (c) o erro irredutível
           label=r"$\sigma^2$")
ax.set_yscale("log")
ax.set_xlabel("grau do polinômio")
ax.set_ylabel("erro quadrático médio")
ax.set_xticks(graus)
ax.legend()
fig.tight_layout()

Deve imprimir `menor erro de teste: grau 3  (0.6044)`.

E aqui vem a surpresa da lista. A nota de aula diz que o mínimo do risco está no
**grau 5** — e nesta amostra ele caiu no **grau 3**. Os valores medidos foram:

| grau | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|
| teste | 1,0338 | 1,0413 | **0,6044** | 0,6110 | 0,6894 | 0,7240 | 1,3421 |

Os graus 3, 4 e 5 estão praticamente empatados, e qual deles ganha depende da
amostra que você sorteou. O Exercício 4 mede exatamente isso.

---
## Exercício 4 — a escolha é confiável?

O Exercício 3 escolheu um grau a partir de **uma** amostra de treino. Repita o
experimento 200 vezes, com amostras de treino independentes, e olhe duas coisas
diferentes:

- o risco **médio** de cada grau — é o que a figura da nota mostra;
- em quantas das 200 amostras cada grau foi o vencedor.

Como conhecemos $r$, dá para calcular o risco sem sortear conjunto de teste:

$$R(\widehat r) = \mathbb{E}\big[(\widehat r(X) - r(X))^2\big] + \sigma^2,$$

bastando percorrer uma grade fina de $x$.

In [ ]:
rng = np.random.default_rng(2026)
x0 = np.linspace(A, B, 500)
r0 = r(x0)
X0 = x0.reshape(-1, 1)

riscos = np.zeros((200, len(graus)))
for b in range(200):
    x, y = amostra(50, rng)
    for j, grau in enumerate(graus):
        modelo = modelo_poly(grau).fit(x.reshape(-1, 1), y)
        riscos[b, j] = np.mean((r0 - modelo.predict(X0)) ** 2) + SIGMA ** 2   # (a)

risco_medio = riscos.mean(axis=0)                           # (b) media sobre as amostras
print("melhor grau, em risco medio:", graus[int(np.argmin(risco_medio))])
for grau, v in zip(graus, risco_medio):
    print(f"  grau {grau:2d}: {v:.4f}")

Deve imprimir `melhor grau, em risco medio: 5`, com

```
  grau  1: 1.0081     grau  6: 0.6645
  grau  2: 1.0452     grau  7: 0.6910
  grau  3: 0.6138     grau  8: 1.5130
  grau  4: 0.6429     grau  9: 2.8063
  grau  5: 0.5838     grau 10: 22.2278
```

Na média, o grau 5 é o melhor — o número da nota se reproduz. O grau 3, que tinha
vencido na amostra do Exercício 3, aparece aqui em terceiro.

Agora a segunda leitura: em cada uma das 200 amostras, qual grau venceu?

In [ ]:
vencedor = graus[np.argmin(riscos, axis=1)]                 # (a) o melhor grau de CADA amostra

for grau in graus:
    quantas = int((vencedor == grau).sum())                 # (b)
    if quantas:
        print(f"grau {grau:2d}: venceu em {quantas:3d}/200  ({100 * quantas / 200:.1f}%)")

print(f"\ngrau 5 NAO foi o melhor em {100 * (vencedor != 5).mean():.1f}% das amostras")

Deve imprimir:

```
grau  3: venceu em  15/200  (7.5%)
grau  4: venceu em   3/200  (1.5%)
grau  5: venceu em 137/200  (68.5%)
grau  6: venceu em  25/200  (12.5%)
grau  7: venceu em  13/200  (6.5%)
grau  8: venceu em   3/200  (1.5%)
grau  9: venceu em   2/200  (1.0%)
grau 10: venceu em   2/200  (1.0%)

grau 5 NAO foi o melhor em 31.5% das amostras
```

O grau 5 é o melhor na média e ainda assim **perde em quase um terço das
amostras** — e o grau 3, que ganhou no Exercício 3, é o vencedor em 7,5% delas.
Ou seja: a sua amostra não era estranha, era um caso comum.

A conclusão é a que abre a Aula 03. Escolher hiperparâmetro olhando o erro de um
conjunto de teste é uma decisão **ruidosa**, e a única saída é reaproveitar os
dados de forma mais eficiente — que é o que a validação cruzada faz. E note o
agravante: se você escolhe o grau pelo conjunto de teste, aquele conjunto deixa
de ser teste, e o erro que ele reporta vira otimista também.

> **Sua vez.** Repita a última célula com amostras de treino de $n=200$ em vez de
> $n=50$. A fração de amostras em que o grau 5 vence sobe ou desce? Era o que você
> esperava?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | o desvio-padrão de $Y$ (1,12) é bem maior que $\sigma$ (0,70): há sinal a estimar |
| 2 | o grau 15 tem o **menor** erro de treino e um erro de teste cinco ordens de grandeza maior |
| 3 | nesta amostra o melhor grau foi o **3**, não o 5 que a nota reporta |
| 4 | na média sobre 200 amostras o grau 5 ganha — mas perde em 31,5% delas |

**A seguir.** A Aula 02 traz a família de modelos que domina a prática quando $d$
é grande — regressão linear e suas versões regularizadas — e reencontra o mesmo
balanço por outro caminho. A ferramenta que resolve o problema do Exercício 4 vem
na Aula 03.